In [ ]:
from ultralytics import YOLO
import torch
import yaml
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import glob
import numpy as np

Скрипт по сохранению каждого N-ного кадра из видео

In [ ]:
import os
import cv2
import glob
from pathlib import Path

# --- НАСТРОЙКИ ---
INPUT_DIR = "/wrk/data/full_data/people_raw_video/video"       # Директория с исходными видео
OUTPUT_DIR = "/wrk/data/full_data/people_raw_video/images"      # Дирекория для сохранения кадров
FRAMES_TO_SAVE = 4           # Количество кадров, которые нужно сохранить
SECONDS_TO_SKIP = 4          # Сколько секунд отрезать от начала
SUPPORTED_EXTENSIONS = ['*.mp4', '*.avi', '*.mov', '*.mkv', '*.webm'] # Поддерживаемые форматы
# -----------------

def get_video_files(directory):
    video_files = []
    for ext in SUPPORTED_EXTENSIONS:
        video_files.extend(glob.glob(os.path.join(directory, ext)))
        video_files.extend(glob.glob(os.path.join(directory, ext.upper())))
    return sorted(video_files)

def process_video(video_path, output_dir):
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"[Ошибка] Не удалось открыть видео: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if fps == 0:
        print(f"[Предупреждение] Не удалось определить FPS для {video_path}. Пропуск.")
        cap.release()
        return

    start_frame_index = int(SECONDS_TO_SKIP * fps)
    
    if start_frame_index >= total_frames:
        print(f"[Пропуск] Видео слишком короткое: {os.path.basename(video_path)}")
        cap.release()
        return

    remaining_frames = total_frames - start_frame_index
    
    if remaining_frames <= 0:
        cap.release()
        return
    
    selected_indices = np.linspace(start_frame_index, total_frames - 1, FRAMES_TO_SAVE, dtype=int)

    video_name = Path(video_path).stem
    created_count = 0

    for target_idx in selected_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
        ret, frame = cap.read()
        
        if ret:
            output_filename = f"{video_name}_frame_{created_count + 1}.jpg"
            output_path = os.path.join(output_dir, output_filename)
            
            cv2.imwrite(output_path, frame)
            created_count += 1
        else:
            print(f"[Ошибка] Не удалось прочитать кадр {target_idx} из {video_name}")

    cap.release()
    print(f"[Готово] Обработано: {video_name}. Сохранено кадров: {created_count}")

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    video_files = get_video_files(INPUT_DIR)
    
    if not video_files:
        print(f"Видеофайлы не найдены в директории: {INPUT_DIR}")
        return

    print(f"Найдено видеофайлов: {len(video_files)}")
    
    for video_path in video_files:
        print(f"Обработка: {os.path.basename(video_path)}...")
        process_video(video_path, OUTPUT_DIR)

if __name__ == "__main__":
    main()

(После разметки в программе CVAT)  
Скрипт по обрезке полосы (пикселей) справа с каждого изображения

In [ ]:
import os
import cv2
import glob
from pathlib import Path

# --- НАСТРОЙКИ ---
IMAGES_DIR = "/wrk/data/full_data/people_raw_video/images"       # Директория с изображениями
LABELS_DIR = "/wrk/data/full_data/people_raw_video/labels"       # Директория с разметкой (YOLO .txt)
OUTPUT_IMAGES_DIR = "/wrk/data/full_data/people_raw_video/images_final"  # Куда сохранить обрезанные изображения
OUTPUT_LABELS_DIR = "/wrk/data/full_data/people_raw_video/labels_final"  # Куда сохранить скорректированную разметку
CROP_PIXELS = 10              # Сколько пикселей убрать справа
# -----------------

def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def process_image_and_label(img_path, label_path, out_img_path, out_lbl_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"[Ошибка] Не удалось прочитать изображение: {img_path}")
        return

    h, w, _ = img.shape
    
    if w <= CROP_PIXELS:
        print(f"[Предупреждение] Изображение слишком узкое: {img_path}")
        return
        
    cropped_img = img[:, :w - CROP_PIXELS]
    new_w = w - CROP_PIXELS
    
    cv2.imwrite(out_img_path, cropped_img)
    

    if not os.path.exists(label_path):
        with open(out_lbl_path, 'w') as f:
            pass
        return

    try:
        with open(label_path, 'r') as f:
            lines = f.readlines()
    except Exception as e:
        print(f"[Ошибка] Чтение файла разметки {label_path}: {e}")
        with open(out_lbl_path, 'w') as f:
            pass
        return

    new_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        parts = line.split()
        if len(parts) != 5:
            continue
            
        cls, x_c, y_c, bw, bh = map(float, parts)
        
        x_abs = x_c * w
        w_abs = bw * w
        
        # Вычисляем правую границу бокса
        right_edge = x_abs + w_abs / 2
        
        if x_abs > new_w:
            continue
            
        new_x_c = x_abs / new_w
        new_bw = w_abs / new_w
        
        new_line = f"{int(cls)} {new_x_c:.6f} {y_c:.6f} {new_bw:.6f} {bh:.6f}\n"
        new_lines.append(new_line)

    with open(out_lbl_path, 'w') as f:
        f.writelines(new_lines)

def main():
    ensure_dir(OUTPUT_IMAGES_DIR)
    ensure_dir(OUTPUT_LABELS_DIR)
    
    img_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']
    image_files = []
    for ext in img_extensions:
        image_files.extend(glob.glob(os.path.join(IMAGES_DIR, ext)))
        image_files.extend(glob.glob(os.path.join(IMAGES_DIR, ext.upper())))
        
    if not image_files:
        print(f"Изображения не найдены в {IMAGES_DIR}")
        return
        
    print(f"Найдено изображений: {len(image_files)}")
    
    processed_count = 0
    for img_path in image_files:
        filename = Path(img_path).stem
        ext = Path(img_path).suffix
        
        label_filename = filename + ".txt"
        src_label_path = os.path.join(LABELS_DIR, label_filename)
        
        dst_img_path = os.path.join(OUTPUT_IMAGES_DIR, filename + ext)
        dst_lbl_path = os.path.join(OUTPUT_LABELS_DIR, label_filename)
        
        process_image_and_label(img_path, src_label_path, dst_img_path, dst_lbl_path)
        processed_count += 1
        
    print(f"Обработка завершена. Обработано файлов: {processed_count}")

if __name__ == "__main__":
    main()

Найдено изображений: 241
Обработка завершена. Обработано файлов: 241


Скрипт для визуальной проверки корректности разметки (накладывает разметку на изображение)

In [ ]:
import cv2
from pathlib import Path
import random
import os

IMG_DIR = "/wrk/data/full_data/people_raw_video/images_final"
LBL_DIR = "/wrk/data/full_data/people_raw_video/labels_final"
OUT_DIR = "/wrk/data/full_data/people_raw_video/check"

os.makedirs(OUT_DIR, exist_ok=True)

image_paths = list(Path(IMG_DIR).glob("*.*"))

sample_paths = random.sample(image_paths, len(image_paths))

for img_path in sample_paths:
    img = cv2.imread(str(img_path))

    if img is None:
        print(f"Cannot read {img_path}")
        continue

    h, w = img.shape[:2]

    lbl_path = Path(LBL_DIR) / f"{img_path.stem}.txt"

    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                cls, xc, yc, bw, bh = map(float, line.split())

                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)

                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), 2)

                cv2.putText(
                    img,
                    f"{int(cls)}",
                    (x1, max(20, y1)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 255, 255),
                    2
                )

    out_path = Path(OUT_DIR) / img_path.name
    cv2.imwrite(str(out_path), img)

print(f"Saved debug images to: {OUT_DIR}")

Saved debug images to: /wrk/data/full_data/people_raw_video/check


Скрипт по сохранению каждого n-ного кадра директории (для прореживания)

In [ ]:
import os
import shutil
from pathlib import Path

# --- НАСТРОЙКИ ---
INPUT_DIR = "/wrk/data/full_data/raw/no_need_crop_need_split"        # Директория с исходными изображениями
OUTPUT_DIR = "/wrk/data/full_data/1_step_after_split/no_need_crop_no_need_split" # Директория для отобранных изображений
STEP = 10                     # Оставлять каждое N-ное изображение (например, 10)
# -----------------

def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def filter_images(input_dir, output_dir, step):
    ensure_dir(output_dir)
    
    extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}
    
    all_files = os.listdir(input_dir)
    
    image_files = sorted([
        f for f in all_files 
        if Path(f).suffix.lower() in extensions
    ])
    
    if not image_files:
        print(f"Изображения не найдены в {input_dir}")
        return

    print(f"Найдено изображений: {len(image_files)}")
    print(f"Шаг выборки: каждое {step}-ое изображение")
    
    selected_count = 0
    
    for i in range(0, len(image_files), step):
        filename = image_files[i]
        src_path = os.path.join(input_dir, filename)
        dst_path = os.path.join(output_dir, filename)
        
        shutil.copy2(src_path, dst_path)
        selected_count += 1
        
    print(f"Готово. Скопировано {selected_count} изображений в {output_dir}")

if __name__ == "__main__":
    filter_images(INPUT_DIR, OUTPUT_DIR, STEP)

Найдено изображений: 2262
Шаг выборки: каждое 10-ое изображение
Готово. Скопировано 227 изображений в /wrk/data/full_data/1_step_after_split/no_need_crop_no_need_split


Скрипт по соотнесению директории с изображениями и директории с разметкой

In [ ]:
import os
import shutil
from pathlib import Path

# --- НАСТРОЙКИ ---
SRC_IMAGES_DIR = "/wrk/data/full_data/1_step_after_split/need_crop_no_need_split"       # Исходная директория с изображениями
SRC_LABELS_DIR = "/wrk/data/full_data/full_labels"       # Исходная директория с разметкой (.txt)

OUT_IMAGES_DIR = "/wrk/data/full_data/2_step_adding_labels/need_crop_images"  # Куда сохранить отобранные изображения
OUT_LABELS_DIR = "/wrk/data/full_data/2_step_adding_labels/need_crop_labels"  # Куда сохранить отобранную разметку
# -----------------

def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def sync_datasets(src_img_dir, src_lbl_dir, out_img_dir, out_lbl_dir):
    ensure_dir(out_img_dir)
    ensure_dir(out_lbl_dir)

    img_files = {f for f in os.listdir(src_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))}
    lbl_files = {f for f in os.listdir(src_lbl_dir) if f.lower().endswith('.txt')}

    print(f"Найдено изображений: {len(img_files)}")
    print(f"Найдено файлов разметки: {len(lbl_files)}")

    img_names_no_ext = {Path(f).stem for f in img_files}
    lbl_names_no_ext = {Path(f).stem for f in lbl_files}

    common_names = img_names_no_ext.intersection(lbl_names_no_ext)

    print(f"Найдено полных пар (картинка + разметка): {len(common_names)}")

    if not common_names:
        print("Нет совпадений между директориями.")
        return

    copied_count = 0
    for name in common_names:
        src_img_path = None
        for f in img_files:
            if Path(f).stem == name:
                src_img_path = os.path.join(src_img_dir, f)
                break
        
        src_lbl_path = os.path.join(src_lbl_dir, f"{name}.txt")

        if src_img_path and os.path.exists(src_lbl_path):
            dst_img_path = os.path.join(out_img_dir, Path(src_img_path).name)
            dst_lbl_path = os.path.join(out_lbl_dir, f"{name}.txt")

            shutil.copy2(src_img_path, dst_img_path)
            shutil.copy2(src_lbl_path, dst_lbl_path)
            copied_count += 1

    print(f"Готово. Скопировано {copied_count} пар в {out_img_dir} и {out_lbl_dir}")

if __name__ == "__main__":
    sync_datasets(SRC_IMAGES_DIR, SRC_LABELS_DIR, OUT_IMAGES_DIR, OUT_LABELS_DIR)

Найдено изображений: 1052
Найдено файлов разметки: 6075
Найдено полных пар (картинка + разметка): 1052
Готово. Скопировано 1052 пар в /wrk/data/full_data/2_step_adding_labels/need_crop_images и /wrk/data/full_data/2_step_adding_labels/need_crop_labels


Скрипт по обрезке панели задач снизу для каждого изображения

In [ ]:
import os
import cv2
import glob
from pathlib import Path

# --- НАСТРОЙКИ ---
IMAGES_DIR = "/wrk/data/full_data/2_step_adding_labels/need_crop_images"       # Директория с исходными изображениями
LABELS_DIR = "/wrk/data/full_data/2_step_adding_labels/need_crop_labels"       # Директория с исходной разметкой (YOLO .txt)
OUTPUT_IMAGES_DIR = "/wrk/data/full_data/3_step_after_crop/images"  # Куда сохранить обрезанные изображения
OUTPUT_LABELS_DIR = "/wrk/data/full_data/3_step_after_crop/labels"  # Куда сохранить скорректированную разметку
CROP_PIXELS_BOTTOM = 50       # Сколько пикселей убрать снизу
# -----------------

def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def process_image_and_label(img_path, label_path, out_img_path, out_lbl_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"[Ошибка] Не удалось прочитать изображение: {img_path}")
        return

    h, w, _ = img.shape
    
    if h <= CROP_PIXELS_BOTTOM:
        print(f"[Предупреждение] Изображение слишком короткое: {img_path}")
        return
        
    cropped_img = img[:h - CROP_PIXELS_BOTTOM, :]
    new_h = h - CROP_PIXELS_BOTTOM
    
    cv2.imwrite(out_img_path, cropped_img)
    
    if not os.path.exists(label_path):
        with open(out_lbl_path, 'w') as f:
            pass
        return

    try:
        with open(label_path, 'r') as f:
            lines = f.readlines()
    except Exception as e:
        print(f"[Ошибка] Чтение файла разметки {label_path}: {e}")
        with open(out_lbl_path, 'w') as f:
            pass
        return

    new_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        parts = line.split()
        if len(parts) != 5:
            continue
            
        cls, x_c, y_c, bw, bh = map(float, parts)
        
        y_abs = y_c * h
        h_abs = bh * h
        
        if y_abs > new_h:
            continue
        
        new_y_c = y_abs / new_h
        new_bh = h_abs / new_h
        
        new_line = f"{int(cls)} {x_c:.6f} {new_y_c:.6f} {bw:.6f} {new_bh:.6f}\n"
        new_lines.append(new_line)

    with open(out_lbl_path, 'w') as f:
        f.writelines(new_lines)

def main():
    ensure_dir(OUTPUT_IMAGES_DIR)
    ensure_dir(OUTPUT_LABELS_DIR)
    
    img_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']
    image_files = []
    for ext in img_extensions:
        image_files.extend(glob.glob(os.path.join(IMAGES_DIR, ext)))
        image_files.extend(glob.glob(os.path.join(IMAGES_DIR, ext.upper())))
        
    if not image_files:
        print(f"Изображения не найдены в {IMAGES_DIR}")
        return
        
    print(f"Найдено изображений: {len(image_files)}")
    
    processed_count = 0
    for img_path in image_files:
        filename = Path(img_path).stem
        ext = Path(img_path).suffix
        
        label_filename = filename + ".txt"
        src_label_path = os.path.join(LABELS_DIR, label_filename)
        
        dst_img_path = os.path.join(OUTPUT_IMAGES_DIR, filename + ext)
        dst_lbl_path = os.path.join(OUTPUT_LABELS_DIR, label_filename)
        
        process_image_and_label(img_path, src_label_path, dst_img_path, dst_lbl_path)
        processed_count += 1
        
    print(f"Обработка завершена. Обработано файлов: {processed_count}")

if __name__ == "__main__":
    main()

Найдено изображений: 1052
Обработка завершена. Обработано файлов: 1052


Скрипт по созданию пустых txt файлов разметки для фоновых изображений

In [ ]:
from pathlib import Path

def sync_missing_txt(images_dir, labels_dir):
    images_root = Path(images_dir)
    labels_root = Path(labels_dir)

    if not images_root.is_dir():
        print(f"❌ Папка изображений не найдена: {images_root}")
        return

    labels_root.mkdir(parents=True, exist_ok=True)

    image_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff'}
    created_count = 0

    for img_path in images_root.rglob('*'):
        if img_path.is_file() and img_path.suffix.lower() in image_exts:
            rel_path = img_path.relative_to(images_root)
            txt_path = labels_root / rel_path.with_suffix('.txt')

            txt_path.parent.mkdir(parents=True, exist_ok=True)

            if not txt_path.exists():
                txt_path.touch()
                created_count += 1
                print(f"✅ Создан: {txt_path}")

    print(f"\n📊 Итог: создано {created_count} пустых .txt файлов в: {labels_root}")

if __name__ == "__main__":
    sync_missing_txt("/wrk/data/full_data/3_step_after_crop/images", "/wrk/data/full_data/3_step_after_crop/labels")

✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-00001.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-00201.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-00401.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-00601.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-00801.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-01001.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-01201.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-01401.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-01601.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-01801.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-02001.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-02201.txt
✅ Создан: /wrk/data/full_data/3_step_after_crop/labels/fon_rain-02401.txt
✅ Создан: /wrk/data/full_data/3_step_a

Скрипт по нарезке изображений высокого качества на кропы (тайлы) по 640x640 с динамическим перекрытием и удалением разметки на краях изображений, если площадь обрезанной разметки менее 50% общей площади объекта, а также удалением разметки с исходных изображений, если при сжатии внутри YOLO объект будет сжат до 0 пикселей и сохранением оригинала

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

# ================= НАСТРОЙКИ =============================
SRC_IMAGES = "/wrk/data/full_data/3_step_after_crop/images"
SRC_LABELS = "/wrk/data/full_data/3_step_after_crop/labels"

DST_IMAGES = "/wrk/data/full_data/4_step_640x640/images"
DST_LABELS = "/wrk/data/full_data/4_step_640x640/labels"

TILE_SIZE = 640
MIN_SIZE_FOR_TILING = 1920  # Порог, начиная с которого считаем изображение "High Res" и режем его

# Если объект на оригинале меньше этого размера, он считается шумом/незначительным и удаляется из разметки оригинала.
MIN_ORIG_BOX_SIZE_MAP = {
    2000: 10,   # Для ~FHD
    4000: 40,   # Для ~4K
    7680: 64,   # Для ~8K
}

# Параметры для тайлов
MIN_VISIBLE_RATIO = 0.5   # Удалять объект, если на тайле видно менее 50% его площади
MIN_BOX_SIZE_PIXELS = 4   # Минимальный размер бокса в тайле (страховка от артефактов)
SAVE_EMPTY_TILES = True  # Сохранять ли тайлы, где нет ни одного объекта после обрезки

# Диапазон перекрытия (overlap)
MIN_OVERLAP_RATIO = 0.2
MAX_OVERLAP_RATIO = 0.7

os.makedirs(DST_IMAGES, exist_ok=True)
os.makedirs(DST_LABELS, exist_ok=True)

def calculate_overlap_ratio(w, h):
    max_dim = max(w, h)
    
    if max_dim <= MIN_SIZE_FOR_TILING:
        return MIN_OVERLAP_RATIO
    
    k = (MAX_OVERLAP_RATIO - MIN_OVERLAP_RATIO) / (7680 - MIN_SIZE_FOR_TILING)
    overlap = MIN_OVERLAP_RATIO + k * (max_dim - MIN_SIZE_FOR_TILING)
    
    return np.clip(overlap, MIN_OVERLAP_RATIO, MAX_OVERLAP_RATIO)

def get_min_orig_box_size(w, h):
    max_dim = max(w, h)
    sorted_thresholds = sorted(MIN_ORIG_BOX_SIZE_MAP.keys())
    
    for threshold in sorted_thresholds:
        if max_dim <= threshold:
            return MIN_ORIG_BOX_SIZE_MAP[threshold]
    
    return MIN_ORIG_BOX_SIZE_MAP[sorted_thresholds[-1]]

def load_yolo_labels(label_path, img_w, img_h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                cls, xc, yc, w_norm, h_norm = map(float, line.split())
                
                x1 = (xc - w_norm / 2) * img_w
                y1 = (yc - h_norm / 2) * img_h
                x2 = (xc + w_norm / 2) * img_w
                y2 = (yc + h_norm / 2) * img_h
                
                boxes.append([int(cls), x1, y1, x2, y2])
            except ValueError:
                continue
    return boxes

def save_yolo_labels(path, boxes, w, h):
    with open(path, "w") as f:
        for cls, x1, y1, x2, y2 in boxes:
            bw = x2 - x1
            bh = y2 - y1
            
            if bw <= 0 or bh <= 0:
                continue

            xc = (x1 + x2) / 2 / w
            yc = (y1 + y2) / 2 / h
            bw_norm = bw / w
            bh_norm = bh / h

            f.write(f"{cls} {xc:.6f} {yc:.6f} {bw_norm:.6f} {bh_norm:.6f}\n")

def filter_boxes_by_size(boxes, min_size_px):
    filtered = []
    for box in boxes:
        cls, x1, y1, x2, y2 = box
        w = x2 - x1
        h = y2 - y1
        if w >= min_size_px and h >= min_size_px:
            filtered.append(box)
    return filtered

def clip_box_to_tile(box, tx, ty):
    cls, x1, y1, x2, y2 = box
    
    original_area = (x2 - x1) * (y2 - y1)
    if original_area == 0:
        return None

    nx1 = max(x1, tx)
    ny1 = max(y1, ty)
    nx2 = min(x2, tx + TILE_SIZE)
    ny2 = min(y2, ty + TILE_SIZE)

    if nx2 <= nx1 or ny2 <= ny1:
        return None

    clipped_area = (nx2 - nx1) * (ny2 - ny1)
    
    if clipped_area / original_area < MIN_VISIBLE_RATIO:
        return None

    lx1 = nx1 - tx
    ly1 = ny1 - ty
    lx2 = nx2 - tx
    ly2 = ny2 - ty
    
    if (lx2 - lx1) < MIN_BOX_SIZE_PIXELS or (ly2 - ly1) < MIN_BOX_SIZE_PIXELS:
        return None

    return [cls, lx1, ly1, lx2, ly2]

def generate_positions(dim_size, stride):
    if dim_size <= TILE_SIZE:
        return [0]

    positions = []
    pos = 0
    while pos + TILE_SIZE <= dim_size:
        positions.append(pos)
        pos += stride

    if positions and positions[-1] + TILE_SIZE < dim_size:
        positions.append(dim_size - TILE_SIZE)
        
    return positions

def process_image(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"Could not read {img_path}")
        return

    h, w = img.shape[:2]
    label_path = Path(SRC_LABELS) / f"{img_path.stem}.txt"
    
    original_boxes = load_yolo_labels(str(label_path), w, h)
    
    stem = img_path.stem
    
    if max(w, h) < MIN_SIZE_FOR_TILING:
        out_name = f"{stem}"
        cv2.imwrite(str(Path(DST_IMAGES) / f"{out_name}.jpg"), img)
        save_yolo_labels(str(Path(DST_LABELS) / f"{out_name}.txt"), original_boxes, w, h)
        return

    overlap_ratio = calculate_overlap_ratio(w, h)
    stride = int(TILE_SIZE * (1 - overlap_ratio))
    
    if stride < 1:
        stride = 1
        
    min_box_px = get_min_orig_box_size(w, h)
    filtered_orig_boxes = filter_boxes_by_size(original_boxes, min_box_px)
    
    orig_out_name = f"{stem}_orig"
    cv2.imwrite(str(Path(DST_IMAGES) / f"{orig_out_name}.jpg"), img)
    save_yolo_labels(str(Path(DST_LABELS) / f"{orig_out_name}.txt"), filtered_orig_boxes, w, h)

    x_positions = generate_positions(w, stride)
    y_positions = generate_positions(h, stride)

    tile_id = 0
    for y in y_positions:
        for x in x_positions:
            tile_img = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
            
            tile_boxes = []
            for box in original_boxes:
                clipped_box = clip_box_to_tile(box, x, y)
                if clipped_box is not None:
                    tile_boxes.append(clipped_box)
            
            if not SAVE_EMPTY_TILES and len(tile_boxes) == 0:
                continue
            
            tile_name = f"{stem}_tile_{tile_id:04d}"
            cv2.imwrite(str(Path(DST_IMAGES) / f"{tile_name}.jpg"), tile_img)
            save_yolo_labels(str(Path(DST_LABELS) / f"{tile_name}.txt"), tile_boxes, TILE_SIZE, TILE_SIZE)
            
            tile_id += 1

def main():
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp']
    image_paths = []
    for ext in extensions:
        image_paths.extend(Path(SRC_IMAGES).glob(ext))
        image_paths.extend(Path(SRC_IMAGES).glob(ext.upper()))
        
    image_paths = list(set(image_paths))
    
    print(f"Found {len(image_paths)} images in {SRC_IMAGES}")
    print(f"Output dir: {DST_IMAGES}")
    print(f"Tiling threshold: {MIN_SIZE_FOR_TILING}px")
    print(f"Overlap range: {MIN_OVERLAP_RATIO} - {MAX_OVERLAP_RATIO}")

    for i, img_path in enumerate(image_paths):
        process_image(img_path)
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(image_paths)}")

    print("DONE")

if __name__ == "__main__":
    main()

Found 1380 images in /wrk/data/full_data/3_step_after_crop/images
Output dir: /wrk/data/full_data/4_step_640x640/images
Tiling threshold: 1920px
Overlap range: 0.2 - 0.7
Processed 50/1380
Processed 100/1380
Processed 150/1380
Processed 200/1380
Processed 250/1380
Processed 300/1380
Processed 350/1380
Processed 400/1380
Processed 450/1380
Processed 500/1380
Processed 550/1380
Processed 600/1380
Processed 650/1380
Processed 700/1380
Processed 750/1380
Processed 800/1380
Processed 850/1380
Processed 900/1380
Processed 950/1380
Processed 1000/1380
Processed 1050/1380
Processed 1100/1380
Processed 1150/1380
Processed 1200/1380
Processed 1250/1380
Processed 1300/1380
Processed 1350/1380
DONE


Скрипт по подсчету количества пустых изображений (фона)

In [ ]:
import os
from pathlib import Path

# --- НАСТРОЙКИ ---
LABELS_DIR = "/wrk/data/full_data/4_step_640x640/labels" # Директория с разметкой
# -----------------

def count_images_with_objects(labels_dir):
    label_files = list(Path(labels_dir).glob("*.txt"))
    
    total = len(label_files)
    empty_count = 0
    with_objects_count = 0
    
    for lbl_path in label_files:
        if lbl_path.stat().st_size == 0:
            empty_count += 1
        else:
            with open(lbl_path, 'r') as f:
                content = f.read().strip()
                if not content:
                    empty_count += 1
                else:
                    with_objects_count += 1
                    
    print(f"Всего файлов разметки: {total}")
    print(f"С объектами: {with_objects_count}")
    print(f"Пустых (без объектов): {empty_count}")

if __name__ == "__main__":
    count_images_with_objects(LABELS_DIR)

Всего файлов разметки: 29765
С объектами: 6089
Пустых (без объектов): 23676


Скрипт по удалению n-ного количества файлов (изображений + разметки)

In [ ]:
import os
import random
from pathlib import Path

# --- НАСТРОЙКИ ---
IMAGES_DIR = "/wrk/data/full_data/FULL_DATA/images"
LABELS_DIR = "/wrk/data/full_data/FULL_DATA/labels"
N_TO_REMOVE = 14600  # Сколько файлов без объектов нужно удалить
# -----------------

def find_empty_pairs(images_dir, labels_dir):
    labels_path = Path(labels_dir)
    images_path = Path(images_dir)
    
    empty_pairs = []
    
    for lbl_file in labels_path.glob("*.txt"):
        is_empty = False
        
        if lbl_file.stat().st_size == 0:
            is_empty = True
        else:
            with open(lbl_file, 'r') as f:
                if not f.read().strip():
                    is_empty = True
        
        if is_empty:
            stem = lbl_file.stem
            img_file = None
            
            for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']:
                candidate = images_path / f"{stem}{ext}"
                if candidate.exists():
                    img_file = candidate
                    break
            
            if img_file:
                empty_pairs.append((img_file, lbl_file))
            else:
                print(f"Предупреждение: Найдена пустая разметка {stem}.txt, но изображение не найдено.")
                
    return empty_pairs

def remove_n_random_empty(n):
    images_path = Path(IMAGES_DIR)
    labels_path = Path(LABELS_DIR)
    
    if not images_path.exists() or not labels_path.exists():
        print("Ошибка: Одна из директорий не найдена.")
        return

    empty_pairs = find_empty_pairs(IMAGES_DIR, LABELS_DIR)
    total_empty = len(empty_pairs)
    
    print(f"Найдено всего файлов без объектов: {total_empty}")
    
    if total_empty == 0:
        print("Нет файлов для удаления.")
        return
    
    if n > total_empty:
        print(f"Запрошено удаление {n}, но найдено только {total_empty}. Будут удалены все найденные.")
        n = total_empty
    
    to_delete = random.sample(empty_pairs, n)
    
    print(f"\nПлан удаления ({n} файлов):")
    for img, lbl in to_delete[:5]:
        print(f"  - {img.name}")
    if n > 5:
        print(f"  ... и еще {n-5} файлов")
    
    confirm = input(f"\nВы уверены, что хотите БЕЗВОЗВРАТНО удалить эти {n} пар? (yes/no): ")
    if confirm.lower() != 'yes':
        print("Операция отменена.")
        return
    
    deleted_count = 0
    for img_path, lbl_path in to_delete:
        try:
            lbl_path.unlink()
            img_path.unlink()
            deleted_count += 1
        except Exception as e:
            print(f"Ошибка при удалении {img_path.name}: {e}")
            
    print(f"\nГотово. Удалено {deleted_count} фоновых пар.")

if __name__ == "__main__":
    random.seed(42) 
    remove_n_random_empty(N_TO_REMOVE)

Найдено всего файлов без объектов: 23676

План удаления (14600 файлов):
  - fon_sumerki-01001_tile_0025.jpg
  - 2026-04-17 14-19-18_frame_3_tile_0030.jpg
  - 2026-04-17 12-03-39_frame_2_tile_0000.jpg
  - 2026-04-17 15-59-55_frame_3_tile_0018.jpg
  - 2026-04-17 15-39-59_frame_2_tile_0023.jpg
  ... и еще 14595 файлов

Готово. Удалено 14600 фоновых пар.


Скрипт по разделению выборки на train/val со стратификацией

In [ ]:
import os
import shutil
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

# --- НАСТРОЙКИ ---
SRC_IMAGES_DIR = "/wrk/data/full_data/FULL_DATA/images"
SRC_LABELS_DIR = "/wrk/data/full_data/FULL_DATA/labels"

OUTPUT_ROOT = "/wrk/data/full_data/FULL_DATA"
TEST_SIZE = 0.2
RANDOM_STATE = 42
# -----------------

def get_image_classes(label_path):
    if not os.path.exists(label_path) or os.path.getsize(label_path) == 0:
        return [-1]
    
    classes = set()
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                try:
                    cls = int(float(parts[0]))
                    classes.add(cls)
                except:
                    continue
    return list(classes) if classes else [-1]

def smart_split():
    images_path = Path(SRC_IMAGES_DIR)
    labels_path = Path(SRC_LABELS_DIR)
    
    img_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    
    image_files = []
    labels_map = {}
    
    for f in os.listdir(images_path):
        if Path(f).suffix.lower() in img_extensions:
            stem = Path(f).stem
            image_files.append(f)
            labels_map[stem] = f

    if not image_files:
        print("Изображения не найдены!")
        return

    print(f"Всего изображений: {len(image_files)}")
    
    stratify_labels = []
    valid_images = []
    
    for img_name in image_files:
        stem = Path(img_name).stem
        lbl_path = labels_path / f"{stem}.txt"
        classes = get_image_classes(lbl_path)

        primary_class = classes[0] 
        stratify_labels.append(primary_class)
        valid_images.append(img_name)

    train_imgs, test_imgs = train_test_split(
        valid_images, 
        test_size=TEST_SIZE, 
        random_state=RANDOM_STATE, 
        stratify=stratify_labels
    )
    
    print(f"Train size: {len(train_imgs)}, Test size: {len(test_imgs)}")
    
    def copy_files(img_list, split_name):
        img_dst = Path(OUTPUT_ROOT) / split_name / "images"
        lbl_dst = Path(OUTPUT_ROOT) / split_name / "labels"
        img_dst.mkdir(parents=True, exist_ok=True)
        lbl_dst.mkdir(parents=True, exist_ok=True)
        
        for img_name in img_list:
            stem = Path(img_name).stem
            src_img = images_path / img_name
            src_lbl = labels_path / f"{stem}.txt"
            
            shutil.copy2(src_img, img_dst / img_name)
            
            if src_lbl.exists():
                shutil.copy2(src_lbl, lbl_dst / f"{stem}.txt")
            else:
                (lbl_dst / f"{stem}.txt").touch()

    copy_files(train_imgs, "train")
    copy_files(test_imgs, "val")
    
    print("Готово! Датасет разделен с учетом баланса классов.")

if __name__ == "__main__":
    smart_split()

Всего изображений: 15165
Train size: 12132, Test size: 3033
Готово! Датасет разделен с учетом баланса классов.


Скрипт по смене id класса

In [ ]:
import os
import glob


labels_dir = "/wrk/data/full_data/FULL_DATA/full/labels"

for txt_file in glob.glob(os.path.join(labels_dir, "*.txt")):
    with open(txt_file, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        class_id = int(parts[0])
        # Маппинг: 0->0, 1->1, 6->3
        if class_id == 6:
            class_id = 3
        
        parts[0] = str(class_id)
        new_lines.append(" ".join(parts) + "\n")
    
    with open(txt_file, 'w') as f:
        f.writelines(new_lines)

print("Готово!")

Готово!
